# CASMI 2026 — CPU retrieval validation

Builds two complementary checks: natural-product source holdout and strict metric-structure holdout. The structure split canonicalizes each distinct training structure with the competition-pinned RDKit, then excludes every raw tautomer key for held-out metric structures from retrieval. The visible test file is not used for model selection.


In [ ]:
from pathlib import Path
import subprocess
import sys

INPUT = Path('/kaggle/input')
DATA_DIR = INPUT / 'competitions/enveda-CASMI26-molecule-id-mass-spectra'
if not (DATA_DIR / 'train.parquet').exists():
    raise FileNotFoundError(f'Competition train.parquet not found under {DATA_DIR}')
source_files = list(INPUT.glob('**/prepare_validation.py'))
if not source_files:
    raise FileNotFoundError('Attach the updated CASMI 2026 Source Code dataset')
SOURCE_ROOT = source_files[0].parent
SRC = SOURCE_ROOT
sys.path.insert(0, str(SRC))
print('Data:', DATA_DIR)
print('Source:', SOURCE_ROOT)
commit_file = SOURCE_ROOT / 'SOURCE_COMMIT.txt'
print('Source version:', commit_file.read_text().strip() if commit_file.exists() else 'missing provenance file')

try:
    import rdkit
    parts = tuple(int(p) for p in rdkit.__version__.split('.')[:3])
except (ImportError, ValueError):
    parts = None
if parts != (2026, 3, 3):
    wheels = [p for p in INPUT.glob('**/*.whl') if 'rdkit' in p.name.lower() or '2026.03.3' in str(p)]
    if not wheels:
        raise RuntimeError('Attach the offline RDKit 2026.03.3 wheel dataset')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--upgrade', str(wheels[0])], check=True)
    import rdkit
    print('RDKit:', rdkit.__version__)


In [ ]:
import subprocess

TRAIN = str(DATA_DIR / 'train.parquet')
WORK = Path('/kaggle/working/validation')
WORK.mkdir(parents=True, exist_ok=True)
source_dir = WORK / 'natural_product_source'
subprocess.run([
    sys.executable, str(SRC / 'prepare_validation.py'), '--train', TRAIN,
    '--output-dir', str(source_dir), '--mode', 'source',
    '--query-source', 'enveda-np-examples',
], check=True)


In [ ]:
# Strict structure holdout. The metric-key map is cached in WORK so reruns reuse it.
structure_dir = WORK / 'structure_heldout'
subprocess.run([
    sys.executable, str(SRC / 'prepare_validation.py'), '--train', TRAIN,
    '--output-dir', str(structure_dir), '--mode', 'structure',
    '--fraction', '0.001', '--workers', '4', '--seed', '20260929',
], check=True)


In [ ]:
def score_holdout(folder, prediction_name, *, exclude_source=None, exclude_keys_file=None):
    predictions = WORK / prediction_name
    command = [
        sys.executable, str(SRC / 'retrieve.py'),
        '--train', TRAIN,
        '--test', str(folder / 'queries.parquet'),
        '--output', str(predictions), '--max-ppm', '30',
        '--fallback-ppm', '250', '--top-k', '25',
    ]
    if exclude_source:
        command.extend(['--exclude-source', exclude_source])
    if exclude_keys_file:
        command.extend(['--exclude-keys-file', str(exclude_keys_file)])
    subprocess.run(command, check=True)
    subprocess.run([
        sys.executable, str(SRC / 'evaluate_predictions.py'),
        '--queries', str(folder / 'queries.parquet'),
        '--predictions', str(predictions),
    ], check=True)

score_holdout(
    source_dir, 'natural_product_source_predictions.csv',
    exclude_source='enveda-np-examples',
)
score_holdout(
    structure_dir, 'structure_heldout_predictions.csv',
    exclude_keys_file=structure_dir / 'heldout_raw_keys.txt',
)


Save both metric outputs with the source commit. Source holdout estimates retrieval on a known structure from another library; structure holdout tests generalization when the molecule itself is absent from the reference. Treat them as separate validation regimes, not as hidden leaderboard estimates.
